## Nodes

In [1]:
import itertools
from collections import deque
import pandas as pd

CANDS = 'ABC'
ORDERS = list(itertools.permutations(CANDS))       # 6 vote-count orderings, least -> most
PAIRS = [('A', 'B'), ('B', 'C'), ('A', 'C')]        # the 3 pairwise matchups
RANK = {'A': 2, 'B': 1, 'C': 0}                     # coalition preference A > B > C

def tournaments():
    for bits in itertools.product([True, False], repeat=3):
        yield frozenset((x, y) if win else (y, x) for (x, y), win in zip(PAIRS, bits))

TOURNAMENTS = list(tournaments())                   # 8 tournament structures T
NODES = [(order, T) for order in ORDERS for T in TOURNAMENTS]   # P = <order | T>

len(ORDERS), len(TOURNAMENTS), len(NODES)

(6, 8, 48)

In [2]:
def winner(node):
    # f(P): the pairwise winner between the top two vote-getters
    order, T = node
    y, z = order[1], order[2]
    return y if (y, z) in T else z

def fmt_T(T):
    return ','.join(f'{x}>{y}' if (x, y) in T else f'{y}>{x}' for x, y in PAIRS)

def fmt_node(node):
    order, T = node
    return f"<{'<'.join(order)} | {fmt_T(T)}>"

## IRV and STAR graphs

In [3]:
def swap(order, i, j):
    lst = list(order)
    lst[i], lst[j] = lst[j], lst[i]
    return tuple(lst)

def irv_moves(node):
    # any adjacent swap in scores, unless it's moving A up
    order, T = node
    out = []
    for i in (0, 1):
        if order[i] == 'A':
            continue
        out.append((swap(order, i, i + 1), T))
    return out

def star_moves(node):
    # any adjacent swap in scores, unless it's moving A above C
    order, T = node
    out = []
    for i in (0, 1):
        j = i + 1
        if order[i] == 'A' and order[j] == 'C':
            continue
        out.append((swap(order, i, j), T))
    return out

irv_graph = {n: irv_moves(n) for n in NODES}
star_graph = {n: star_moves(n) for n in NODES}

sum(len(v) for v in irv_graph.values()), sum(len(v) for v in star_graph.values())

(64, 80)

## Simple manipulations (A≻B≻C coalition)

In [4]:
def move_desc(node, move):
    order, new_order = node[0], move[0]
    i, j = [k for k in range(3) if order[k] != new_order[k]]
    return f"{order[i]} moves up, past {order[j]}"

COLS_SIMPLE = ['start', 'move', 'end', 'f(start)', 'f(end)']

def has_simple_manipulation(node, graph):
    f0 = winner(node)
    return any(RANK[winner(mv)] > RANK[f0] for mv in graph[node])

def simple_manipulations(graph):
    rows = []
    for node in NODES:
        f0 = winner(node)
        for move in graph[node]:
            f1 = winner(move)
            if RANK[f1] > RANK[f0]:      # strictly better for the coalition
                rows.append({'start': fmt_node(node), 'move': move_desc(node, move),
                             'end': fmt_node(move), 'f(start)': f0, 'f(end)': f1})
    return pd.DataFrame(rows, columns=COLS_SIMPLE)

irv_simple_df = simple_manipulations(irv_graph)
irv_simple_df

,start,move,end,f(start),f(end)
0,"<B<A<C | A>B,B>C,C>A>","B moves up, past A","<A<B<C | A>B,B>C,C>A>",C,B
1,"<B<A<C | B>A,B>C,C>A>","B moves up, past A","<A<B<C | B>A,B>C,C>A>",C,B
2,"<B<C<A | A>B,B>C,C>A>","B moves up, past C","<C<B<A | A>B,B>C,C>A>",C,A
3,"<B<C<A | A>B,C>B,C>A>","B moves up, past C","<C<B<A | A>B,C>B,C>A>",C,A
4,"<B<C<A | B>A,B>C,C>A>","B moves up, past C","<C<B<A | B>A,B>C,C>A>",C,B
5,"<B<C<A | B>A,C>B,C>A>","B moves up, past C","<C<B<A | B>A,C>B,C>A>",C,B
6,"<C<B<A | B>A,B>C,A>C>","C moves up, past B","<B<C<A | B>A,B>C,A>C>",B,A
7,"<C<B<A | B>A,C>B,A>C>","C moves up, past B","<B<C<A | B>A,C>B,A>C>",B,A


In [5]:
star_simple_df = simple_manipulations(star_graph)
star_simple_df

,start,move,end,f(start),f(end)
0,"<A<B<C | A>B,B>C,A>C>","A moves up, past B","<B<A<C | A>B,B>C,A>C>",B,A
1,"<A<B<C | A>B,C>B,A>C>","A moves up, past B","<B<A<C | A>B,C>B,A>C>",C,A
2,"<A<B<C | B>A,B>C,A>C>","A moves up, past B","<B<A<C | B>A,B>C,A>C>",B,A
3,"<A<B<C | B>A,C>B,A>C>","A moves up, past B","<B<A<C | B>A,C>B,A>C>",C,A
4,"<B<A<C | A>B,B>C,C>A>","B moves up, past A","<A<B<C | A>B,B>C,C>A>",C,B
5,"<B<A<C | B>A,B>C,C>A>","B moves up, past A","<A<B<C | B>A,B>C,C>A>",C,B
6,"<B<C<A | A>B,B>C,C>A>","B moves up, past C","<C<B<A | A>B,B>C,C>A>",C,A
7,"<B<C<A | A>B,C>B,C>A>","B moves up, past C","<C<B<A | A>B,C>B,C>A>",C,A
8,"<B<C<A | B>A,B>C,C>A>","B moves up, past C","<C<B<A | B>A,B>C,C>A>",C,B
9,"<B<C<A | B>A,C>B,C>A>","B moves up, past C","<C<B<A | B>A,C>B,C>A>",C,B


## Complex manipulations

In [6]:
def shortest_multistep_path(start, graph):
    # shortest walk of >= 2 safe steps (every node visited weakly preferred,
    # >=, to f(start)) ending at a node strictly preferred to f(start).
    # A direct 1-step (simple) manipulation, if one exists, is ignored here
    # on purpose -- see the `has simple too` column below.
    f0 = winner(start)
    dist = {start: 0}
    prev = {start: None}
    queue = deque([start])
    while queue:
        cur = queue.popleft()
        for nxt in graph[cur]:
            if nxt in dist:
                continue
            f1 = winner(nxt)
            if RANK[f1] < RANK[f0]:
                continue              # would regress below the start, not a safe step
            dist[nxt] = dist[cur] + 1
            prev[nxt] = cur
            if RANK[f1] > RANK[f0] and dist[nxt] >= 2:
                path, p = [nxt], cur
                while p is not None:
                    path.append(p)
                    p = prev[p]
                return path[::-1]
            queue.append(nxt)
    return None

COLS_COMPLEX = ['start', 'path', 'steps', 'f(start)', 'f(end)', 'has simple too']

def complex_manipulations(graph):
    # every state with a profitable walk of 2+ safe steps, whether or not
    # a shorter (simple) manipulation is *also* available from it
    rows = []
    for node in NODES:
        path = shortest_multistep_path(node, graph)
        if path is None:
            continue
        rows.append({'start': fmt_node(path[0]),
                     'path': ' -> '.join(fmt_node(n) for n in path),
                     'steps': len(path) - 1,
                     'f(start)': winner(path[0]), 'f(end)': winner(path[-1]),
                     'has simple too': has_simple_manipulation(node, graph)})
    return pd.DataFrame(rows, columns=COLS_COMPLEX)

irv_complex_df = complex_manipulations(irv_graph)
irv_complex_df

,start,path,steps,f(start),f(end),has simple too
0,"<B<A<C | A>B,B>C,C>A>","<B<A<C | A>B,B>C,C>A> -> <A<B<C | A>B,B>C,C>A>...",2,C,B,True
1,"<B<A<C | B>A,B>C,C>A>","<B<A<C | B>A,B>C,C>A> -> <A<B<C | B>A,B>C,C>A>...",2,C,B,True
2,"<B<C<A | A>B,B>C,C>A>","<B<C<A | A>B,B>C,C>A> -> <C<B<A | A>B,B>C,C>A>...",2,C,A,True
3,"<B<C<A | A>B,C>B,C>A>","<B<C<A | A>B,C>B,C>A> -> <C<B<A | A>B,C>B,C>A>...",2,C,A,True
4,"<B<C<A | B>A,B>C,C>A>","<B<C<A | B>A,B>C,C>A> -> <C<B<A | B>A,B>C,C>A>...",2,C,B,True
5,"<B<C<A | B>A,C>B,C>A>","<B<C<A | B>A,C>B,C>A> -> <C<B<A | B>A,C>B,C>A>...",2,C,B,True
6,"<C<B<A | B>A,B>C,A>C>","<C<B<A | B>A,B>C,A>C> -> <B<C<A | B>A,B>C,A>C>...",2,B,A,True
7,"<C<B<A | B>A,C>B,A>C>","<C<B<A | B>A,C>B,A>C> -> <B<C<A | B>A,C>B,A>C>...",2,B,A,True


In [7]:
star_complex_df = complex_manipulations(star_graph)
star_complex_df

,start,path,steps,f(start),f(end),has simple too
0,"<A<C<B | A>B,B>C,A>C>","<A<C<B | A>B,B>C,A>C> -> <A<B<C | A>B,B>C,A>C>...",2,B,A,False
1,"<A<C<B | A>B,C>B,A>C>","<A<C<B | A>B,C>B,A>C> -> <A<B<C | A>B,C>B,A>C>...",2,C,A,False
2,"<A<C<B | B>A,B>C,A>C>","<A<C<B | B>A,B>C,A>C> -> <A<B<C | B>A,B>C,A>C>...",2,B,A,False
3,"<A<C<B | B>A,C>B,A>C>","<A<C<B | B>A,C>B,A>C> -> <A<B<C | B>A,C>B,A>C>...",2,C,A,False
4,"<B<A<C | A>B,B>C,C>A>","<B<A<C | A>B,B>C,C>A> -> <A<B<C | A>B,B>C,C>A>...",2,C,B,True
5,"<B<A<C | B>A,B>C,C>A>","<B<A<C | B>A,B>C,C>A> -> <A<B<C | B>A,B>C,C>A>...",2,C,B,True
6,"<B<C<A | A>B,B>C,C>A>","<B<C<A | A>B,B>C,C>A> -> <C<B<A | A>B,B>C,C>A>...",2,C,A,True
7,"<B<C<A | A>B,C>B,C>A>","<B<C<A | A>B,C>B,C>A> -> <C<B<A | A>B,C>B,C>A>...",2,C,A,True
8,"<B<C<A | B>A,B>C,C>A>","<B<C<A | B>A,B>C,C>A> -> <C<B<A | B>A,B>C,C>A>...",2,C,B,True
9,"<B<C<A | B>A,C>B,C>A>","<B<C<A | B>A,C>B,C>A> -> <C<B<A | B>A,C>B,C>A>...",2,C,B,True


## Total manipulable states

In [8]:
# a state is manipulable if it has a simple manipulation, a complex one, or both
def is_manipulable(node, graph):
    return has_simple_manipulation(node, graph) or shortest_multistep_path(node, graph) is not None

irv_manipulable = {n for n in NODES if is_manipulable(n, irv_graph)}
star_manipulable = {n for n in NODES if is_manipulable(n, star_graph)}

# cross-check against the two tables above: manipulable == simple ∪ complex
irv_from_tables = set(irv_simple_df['start']) | set(irv_complex_df['start'])
star_from_tables = set(star_simple_df['start']) | set(star_complex_df['start'])
assert {fmt_node(n) for n in irv_manipulable} == irv_from_tables
assert {fmt_node(n) for n in star_manipulable} == star_from_tables

pd.Series({'IRV': len(irv_manipulable), 'STAR': len(star_manipulable)}, name='manipulable states (of 48)')

IRV      8
STAR    18
Name: manipulable states (of 48), dtype: int64